# NHANES ANALYSIS: Can we identify trends in nutrition, activity levels and income in people with and without diabetes? 

- Name: Tyson Mitchell
  
- ID: 225734851

  
- email: s225734851@deakin.edu.au

  
- SIT731 Student

In this report, we are going to investigate the NHANES database. NHANES is the National Health and Nutrition Examination Survey. This is a national survey that measures health and nutrition of adults and children in the United States. While it is a survey, it also includes health exams and laboratary tests. This data includes information collected from over 5000 participants.

From this database, we are going to explore the relationships between nutrition, physical activity, occupation and diabetes prevalence. This report is going to use data obtained from NHANES August 2021 - August 2023. We will be exploring the difference in nutritional intake between those with diabetes and those without. This will include sugar, fiber and protein consumption, along with total calories. Then, we are going to investigate the actviity levels of those with diabetes and those without. We will be using sedentary levels and activity levels to compare how this differs across the sample populations. Then, we are going to compare diabetes prevelance based on income and occupation type to assess how diabetes is seen in different working populations. 

This report is going to analyse this data that was taken at one point in time. We will be drawing conclusions on which variables are different based on the time of recording, and we will assess if there is a difference in nutritional or activity behaviours between those diagnosed with diabetes and those who are not. 

First, we are going to load the data from NHANES. 

In [ ]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, HoverTool, Select, CustomJS, RangeSlider
from bokeh.layouts import column, row

output_notebook(hide_banner=True)

In [ ]:
df_demo = pd.read_sas('/users/tysonmitchell/Downloads/DEMO_L.xpt') #demographic
df_di1 = pd.read_sas('/users/tysonmitchell/Downloads/DR1TOT_L.xpt') #day 1 nutrition values
df_di2 = pd.read_sas('/users/tysonmitchell/Downloads/DR2TOT_L.xpt') #day 2 nutrition values
df_ghb = pd.read_sas('/users/tysonmitchell/Downloads/GHB_L.xpt') #glucohem
df_bm = pd.read_sas('/users/tysonmitchell/Downloads/BMX_L.xpt') #body measures
df_inc = pd.read_sas('/users/tysonmitchell/Downloads/INQ_L.xpt') #income
df_dbt = pd.read_sas('/users/tysonmitchell/Downloads/DIQ_L.xpt') #diabetes
df_pa = pd.read_sas('/users/tysonmitchell/Downloads/PAQ_L.xpt') #phyiscal activity
df_occ = pd.read_sas('/users/tysonmitchell/Downloads/OCQ_L.xpt') #occupation


df_di2.columns = [col.replace('DR2', 'DR1') if col.startswith('DR2') else col 
                  for col in df_di2.columns]

df_diet = (pd.concat([df_di1, df_di2])
             .groupby('SEQN')
             .mean()
             .reset_index())

As we can see, we have downloaded 10 datasets from the NHANES study:

1. df_demo: demographic variables and sample weights of the participants, including gender, age, race, education level etc. 

2. df_diet: the average values of the total nutrient intakes first day and second day. This comprises of total nutrient intakes for each participant, including type of salt used, weight loss/diet presence, number of food, protein, energy etc.

3. df_ghb: glycohemoglobin: contains phlebotomy and glycohemoglobin percentage. Marker of blood sugar levels.

4. df_bm: Body Measures: data surrounding the bodily measures of participants, including weight, waist circumference and hip circumference.

5. df_inc = Income measures, including family momthly poverty level, family savings and assets.

6. df_dbt: Diabetes questionnaire: personal interview data on diabetes, prediabetes and use of insulin/hypoglycemic medications.

7. df_pa: Physical activity interview data, including frequency and duration of vigorous and light intensity physical activity.

8. df_occ: Occupation interview data, including employment status, number of hours worked, days per week etc.


Now, we are going to merge them into one dataframe. 

In [ ]:
dfs = [df_diet, df_ghb, df_inc, df_bm, df_dbt, df_pa, df_occ]
df_merged = df_demo

for df in dfs:
    df_merged = df_merged.merge(df, on='SEQN', how='left')

print(df_merged.head(5))
print(df_merged.shape)

As we can see, we have 291 columns in our dataframe. In order to make handling our data easier and clearer, we are going to extract just the features that we need for our graphs. We are going to select columns from the demographics, dietary, health outcomes, physical activity and socioeconomic/income datasets. 

In [ ]:
cols = [
    'SEQN',
    # demographics
    'RIDAGEYR',    # age
    'RIAGENDR',    # gender
    # dietary
    'DR1TKCAL',    # calories
    'DR1TPROT',    # protein
    'DR1TFIBE',    # fibre
    'DR1TSUGR',    # sugar
    'DR1TCARB',    # carbohydrates
    # health outcomes
    'LBXGH',       # HbA1c
    'DIQ010',      # diabetes diagnosis
    # physical activity
    'PAD680',      # sedentary minutes per day
    'PAD790Q',     # frequency of moderate activity
    'PAD800',      # minutes per session moderate activity
    'PAD810Q',     # frequency of vigorous activity
    'PAD820',      # minutes per session vigorous activity
    # socioeconomic
    'OCD150',      # occupation type
    # income and assets
    'INDFMPIR',    # annual poverty ratio (already in DEMO)
]

df = df_merged[cols].copy()

Now we have our new df with just the necessary columns we will be using in this report. Now, we are going to start clean our data by remvoing unrealistic values and values that are placeholders for null values. All of the data in the NHANES datasets have already been encoded, and we are going to check for the presence of values that are physiologically impossible, such as a HbA1c over 20% or under 3%.  

In [ ]:
#Checking for NHANES special codes still present
print("\nSuspicious values check:")
print("HbA1c > 20:      ", (df['LBXGH'] > 20).sum())
print("HbA1c < 3:       ", (df['LBXGH'] < 3).sum())
print("PAD680 > 1380:   ", (df['PAD680'] > 1380).sum()) #sedentary minutes (23 hours), over 23 hours is unrealistic
print("PAD680 == 9999:  ", (df['PAD680'] == 9999).sum()) # 9999 is used for null values 
print("DIQ010 == 7:     ", (df['DIQ010'] == 7).sum()) #7 = refused to answer and 9 equals don't know
print("DIQ010 == 9:     ", (df['DIQ010'] == 9).sum())

As we can see, all of our HbA1c, BMI and BXPOSY1 values are within our accepted range, and thus we don't need to clean them. We have several values that either equal refused or are outside our range for sedentary minutes per day, diabetes diagnosis and savings. Now, we will replace these values for nulls and set some boundaries for features such as income and blood pressure. A key feature of the NHANES dataset is the use of placeholders for null values, such as 7, 77, 777, or 9, 99, and 999. These need to be replaced with null values otherwise they will heavily skew our results. 

In [ ]:
# Fix DIQ010 — replace 7 and 9 values with null
df = df[df['DIQ010'].isin([1, 2, 3]) | df['DIQ010'].isna()]

# Income — NHANES caps at 5
df = df[df['INDFMPIR'].between(0, 5) | df['INDFMPIR'].isna()]

# Physical activity — sedentary time - replace don't know values, 9999, and values over 23 hours
df['PAD680'] = df['PAD680'].replace(9999, np.nan)
df['PAD680'] = df['PAD680'].clip(upper=1380)

# Physical activity — frequency columns (PAD790Q, PAD810Q)
# 777 = refused, 999 = don't know for these frequency-of-activity columns
for col in ['PAD790Q', 'PAD810Q']:
    df[col] = df[col].replace([777, 999], np.nan)

# Physical activity — duration columns (PAD800, PAD820) ─ 7777 = refused, 9999 = don't know for duration-in-minutes columns
for col in ['PAD800', 'PAD820']:
    df[col] = df[col].replace([7777, 9999], np.nan)

# Dietary columns — clip at 99.9th percentile, we will only trim extreme upper outliers
dietary_cols = ['DR1TKCAL', 'DR1TPROT', 'DR1TFIBE', 'DR1TSUGR', 'DR1TCARB']
for col in dietary_cols:
    upper = df[col].quantile(0.999)
    df[col] = df[col].clip(upper=upper)
df = df[df['DR1TKCAL'] > 1] #using 1 as there are some calorie values between 0 and 1 in the dataset

df = df.reset_index(drop=True).copy()

Now that we have cleaned our data, we need to create some variables that will be used in our plots. All of the data in the NHANES datasets are encoded, and thus we need to decode some important classification features such as glycemic status, diabetes status, and categorised savings. Using the pandas function cut, we can create categories based on bins, which we will create categories for columns such as glycemic status. By doing these, we are returning some important features back into strings for easier visualisation and classification of data points in our plots. 

We will use the map function for variables where the values are a set of discrete number codes, such as gender, diabetes status, and occupation. We will use cut for when the variable contains continunous numbers, such as age and glycemic status. 

In [ ]:
# creating a glycemic status based on HbA1c clinical thresholds
df['glycemic_status'] = pd.cut(
    df['LBXGH'],
    bins=[0, 5.7, 6.4, 100],
    labels=['Normal', 'Prediabetic', 'Diabetic']
).astype(object) 

# Recreate diabetes status since we just removed the 4 (don't know)
df['diabetes_status'] = df['DIQ010'].map({
    1: 'Diabetic',
    2: 'No diabetes',
    3: 'Borderline'
})


# Gender labels
df['gender'] = df['RIAGENDR'].map({1: 'Male', 2: 'Female'})

# Age groups- will be used in plotting to create bins 
df['age_group'] = pd.cut(df['RIDAGEYR'],
    bins=[0, 17, 34, 49, 64, 120],
    labels=['<18', '18–34', '35–49', '50–64', '65+'])
df['age_group'] = df['age_group'].astype(str) #not stored as integers 

#occupation
occ_map = {1: 'Employed', 2: 'Seeking work', 3: 'Not working', 4: 'Retired', 5: 'Other'}
df['occupation'] = df['OCD150'].map(occ_map)

Now, we are going to highlight observe some relationships between nutritional data and diabetes prevalence. The first plot is going to examine the relationship with carbohydrates and diabetes. In particular, we are going to explore how a diet that is high in sugar compares to a diet high in fibre in terms of diabetes related outcomes, or in this case, HbAIc. HbAIc (glycated hemoglobin) is a measure of the average blood sugar levels from the last 8-12 weeks. This test is a commonly used method of diagnosing type 2 diabetes. 

In [ ]:
from bokeh.transform import factor_cmap

#First, we are going to create a new plot 1 dataframe with just our necessary columns:
#sugar/carbohydrate ratio, HbA1c, glycemic status, gender and age. We will also drop any null rows. 

#this is our proportion of carbohydrates that are sugar variable, first we ensure that we have only carbohydrate values that are over 0, otherwise a 0 result will output infinity 
df = df[df['DR1TCARB'] > 0]
df['sugar_carb_ratio'] = df['DR1TSUGR'] / df['DR1TCARB']

p1_df = df[['sugar_carb_ratio', 'LBXGH', 'glycemic_status', 
            'gender', 'RIDAGEYR']].dropna()

#Now, we need to create ColumnDataSources as this is what is passed through Bokeh
source_p1 = ColumnDataSource(p1_df)

#creating status order and status colours for factor_cmap, and we will pass these into factor_cmap
status_order  = ['Normal', 'Prediabetic', 'Diabetic']
status_colors = ['#2ecc71', '#f39c12', '#e74c3c']

#now using factor_cmap, we will map each column to a colour
mapper = factor_cmap(
    field_name='glycemic_status',
    palette=status_colors,
    factors=status_order
)

#this is the details for our figure, assigning it a width, height and axis/label titles, along with the bokeh tools we will use 
p1 = figure(
    width=700, height=400,
    title='Daily Sugar/Carbohydrate Proportion vs HbA1c by Glycaemic Status',
    x_axis_label='Proportion of Carbohydrates that are Sugar (Sugar/Carbohydrates)',
    y_axis_label='HbA1c (%)',
    tools='pan,wheel_zoom,reset,save',
)

#now we will map this into a scatter plot, using sugar as our x label and HbA1c as y label
p1.scatter(
    x='sugar_carb_ratio',
    y='LBXGH',
    source=source_p1,
    color=mapper,
    alpha=0.5,
    size=6,
    legend_field='glycemic_status',  # tells Bokeh to build legend from this column
    muted_alpha=0.05,
)

#this will add an interactive hover tool, where hovering over a datapoint will reveal more information about the datapoint
p1.add_tools(HoverTool(tooltips=[
    ('Glycaemic status',   '@glycemic_status'),
    ('Proportion of sugar','@sugar_carb_ratio{0.1f}'),
    ('HbA1c (%)',          '@LBXGH{0.1f}'),
    ('Age',                '@RIDAGEYR'),
    ('Gender',             '@gender'),
]))

p1.legend.title        = 'Glycaemic status'
p1.legend.location     = 'top_right'
p1.legend.click_policy = 'mute'
p1.background_fill_color = '#f9f9f9'
p1.grid.grid_line_color  = 'white'

show(p1)

We have chosen to use sugar proportion as a measurement instead of just sugar, as it would be important to consider the proportion of a person's diet that is sugar, compared to just sugar individually. For example, a paritcipant consuming 300g of carbohydrates a day but only 10g of sugar has a very different nutritional profile to one consuming 50g of carbohydrates a day and 40g of sugar. This proportion allows us to see how a diet that consists of a high proportion of sugar compares to a low proportion.

As we can see, there is quite an even distribution in terms of HbA1c and Sugar consumption. We can see that on the extreme side, there are 3 non-diabetic participants that consumed a pure proportion of sugar, and 1 participant with a percentage of 99% sugar consumed. There are mainly only normal and prediabetic participants who consume a sugar proportion of 80-100% total sugar. The most concentrated area of diabetic participants have a sugar proportion of 20-60%. Of this, there are several participants with substantially high HbA1c readings, with the highest being a Female aged 47 with a HbA1c of 17% and a sugar proportion of 28%. As we can see, we can assume that people who are diabetic are consuming the same if not slightly less proportion of carbohydrates that are sugar compared to prediabetic and non-diabetic participants. However, we can not see a strong enough conclusion to make a strong inference. This dataset was collected at once, and is thus a cross-sectional study. Therefore, we can not infer how participant's diet may have affected diabetes, but instead we can determine that particpants who are diabetic are consuming less if not equal proportions of sugar to those non-diabetic, and therefore we can assume they are following guidelines to lower their sugar intake post-diabetes diagnosis. It also must be noted these nutritional intakes are only taken across 2 days, and therefore does not capture an entire picture of nutritional sugar intake and diabetes diagnosis. 

Now, we will investigate how proportions of fiber consumption vary across diabetic and non-diabetic participants. Fiber does not cause a spike in blood sugar, and has been shown to improve glycemic control and weight management in type 2 diabetes patients (https://pmc.ncbi.nlm.nih.gov/articles/PMC11099360/). 

In [ ]:
#First, we are going to create a new plot 2 dataframe with just our necessary columns:
#sugar, HbA1c, glycemic status, gender and age. We will also drop any null rows. 

p2_df = df[['DR1TFIBE', 'LBXGH', 'glycemic_status', 
            'gender', 'RIDAGEYR']].dropna()

#Now, we need to create ColumnDataSources as this is what is passed through Bokeh
source_p2 = ColumnDataSource(p2_df)

#creating status order and status colours for factor_cmap, and we will pass these into factor_cmap
status_order2  = ['Normal', 'Prediabetic', 'Diabetic']
status_colors2 = ['#2ecc71', '#f39c12', '#e74c3c']

#now using factor_cmap, we will map each column to a colour
mapper2 = factor_cmap(
    field_name='glycemic_status',
    palette=status_colors2,
    factors=status_order2,
)

#this is the details for our figure, assigning it a width, height and axis/label titles, along with the bokeh tools we will use 
p2 = figure(
    width=700, height=400,
    title='Daily Fiber Intake (g/day) vs HbA1c by Glycaemic Status',
    x_axis_label='Daily Intake of Fiber (g/day)',
    y_axis_label='HbA1c (%)',
    tools='pan,wheel_zoom,reset,save',
)

#now we will map this into a scatter plot, using sugar as our x label and HbA1c as y label
p2.scatter(
    x='DR1TFIBE',
    y='LBXGH',
    source=source_p2,
    color=mapper2,
    alpha=0.5,
    size=6,
    legend_field='glycemic_status',  # tells Bokeh to build legend from this column
    muted_alpha=0.05,
)

#this will add an interactive hover tool, where hovering over a datapoint will reveal more information about the datapoint
p2.add_tools(HoverTool(tooltips=[
    ('Glycaemic status',    '@glycemic_status'),
    ('Fiber Intake (g/day)','@DR1TFIBE{0.1f}'),
    ('HbA1c (%)',           '@LBXGH{0.1f}'),
    ('Age',                 '@RIDAGEYR'),
    ('Gender',              '@gender'),
]))

p2.legend.title        = 'Glycaemic status'
p2.legend.location     = 'top_right'
p2.legend.click_policy = 'mute'
p2.background_fill_color = '#f9f9f9'
p2.grid.grid_line_color  = 'white'

show(p2)

We have used total fiber intake instead of proportion, as in the context of fiber consumption, a total fiber intake measurement is more accurate. Using proportion of carbohydrates that are fiber can be misleading, as someone may consume a low number of carbohydrates that is also low in fiber, whereas soemone who consumes a high number of carbohydrates and fiber will score lower. Raw fiber intake was therefore used as a more accurate measurement for understanding health outcomes related to fiber intake. 

Given that we observed a slightly decreased / no difference in sugar/carbohydrate proportion, we can infer that diabetic participants are following a lower sugar diet after their diabetes diagnosis. For this fiber intake plot, it seems that there is a difference in fiber consumption compared to non diabetic and diabetic participants. As we can see, there is a higher proportion of non diabetic and even prediabetic participants that consume above 40g of fiber a day compared to diabetic participants. We can see that there is a cluster of diabetic participants between 0 and 20g of fiber per day, and diabetic participants with the highest HbA1c percentages can be seen in this low fiber consumption area. The American Diabetes Association recommends 25-38g of fiber per day, yet most diabetes participants are consuming between 0 and 20g per day. We can not make any assumptions about fiber causing diabetes diagnosis, but we can see that unlike in the sugar plot, it seems that there is very little fiber increase in reaction to a diabetes diagnosis for participants with diabetes. Therefore, we can assume that participants with diabetes have reduced sugar intake in accordance to potential clinical advice, however it seems that there is an absence of increased fiber intake in reaction to a diabetes diagnosis. This suggests that either dietary recommendations following diabetes diagnosis is either incomplete, or participants are struggling to comply with increased fiber recommendations. 

Now, we are going to investigate the difference in calorie and protein consumption in participants with diabetes compared to no diabetes. Studies have showed a relationship between high calorie intake and worsening insulin resistance (https://pmc.ncbi.nlm.nih.gov/articles/PMC4237976/), while protein has been shown to increase satiety to a greater extent than fat and carbohydrates (https://www.sciencedirect.com/science/article/pii/S0002916523236643), and therefore we will explore this relationship. 

In [ ]:
from bokeh.transform import jitter

#we are using proportion of calories that are protein to observe a high protein diet, as high calories likely leads to high protein and raw protein intake is less accurate
# we have also multiplied protein by 4, as protein is measured in grams and 1 gram of protein is 4 calories 
df['protein_calorie_ratio'] = (df['DR1TPROT'] * 4) / df['DR1TKCAL']
df['protein_calorie_ratio'] = df['protein_calorie_ratio'].clip(upper=1.0)
# we are just going to use total calories, protein, diabetes status, gender and age 
p3_df = df[['DR1TKCAL', 'protein_calorie_ratio', 'diabetes_status',
            'gender', 'RIDAGEYR']].dropna().copy()
#for the javascript below, i found that it wouldn't change y unless i created a separate y column in the source. 
p3_df['y'] = p3_df['DR1TKCAL']

source_p3 = ColumnDataSource(p3_df)

gender_mapper = factor_cmap(
    field_name='gender',
    palette=['#3498db', '#e74c3c'],  # blue for male, red for female
    factors=['Male', 'Female']
)

p3 = figure(
    x_range=['No diabetes', 'Borderline', 'Diabetic'],
    width=700, height=400,
    title='Daily Caloric Intake by Diabetes Status and Gender',
    x_axis_label='Diabetes status',
    y_axis_label='Daily energy intake (kcal)',
    tools='pan,wheel_zoom,reset,save',
)



#We are going to use a strip plot. Our x is categorical, while y is contunous. 
p3.scatter(
    x=jitter('diabetes_status', width=0.3, range=p3.x_range), #create a jitter for easier visualisation
    y='y',
    source=source_p3,
    color=gender_mapper,
    alpha=0.4,
    size=5,
    legend_field='gender',
    muted_alpha=0.05,
)

p3.add_tools(HoverTool(tooltips=[
    ('Diabetes status',  '@diabetes_status'),
    ('Gender',           '@gender'),
    ('Calories (kcal)',  '@DR1TKCAL{0.0f}'),
    ('Protein/kcal',     '@protein_calorie_ratio{0.1f}'),
    ('Age',              '@RIDAGEYR'),
]))

#this is our dropdown to switch between calories and protein
select_p3 = Select(
    title='Y-axis nutrient:',
    value='Calories (kcal/day)',
    options=['Calories (kcal/day)', 'Protein/calorie ratio']
)

callback_p3 = CustomJS(
    args=dict(
        source=source_p3,       # the data
        yaxis=p3.yaxis[0],      # the y axis object so we can update its label
        title=p3.title,         # the title object so we can update it
        sel=select_p3,          # the dropdown widget itself
    ),
    code="""
        // 1. get selected value
        const val = sel.value 

        // 2 & 3. swap y data based on selection
        if (val === 'Calories (kcal/day)') {
            source.data['y'] = source.data['DR1TKCAL'].slice()
            yaxis.axis_label = 'Daily energy intake (kcal)'
            title.text = 'Daily Caloric Intake by Diabetes Status and Gender'
        } else {
            source.data['y'] = source.data['protein_calorie_ratio'].slice()
            yaxis.axis_label = 'Protein as a proportion of caloric intake'
            title.text = 'Protein/Calorie Ratio by Diabetes Status and Gender'
        }

        // 5. tell Bokeh to redraw
        source.change.emit()
    """
)
select_p3.js_on_change('value', callback_p3)

p3.legend.title           = 'Gender'
p3.legend.location        = 'top_right'
p3.legend.click_policy    = 'mute'
p3.background_fill_color  = '#f9f9f9'
p3.grid.grid_line_color   = 'white'
p3.xaxis.axis_label_text_font_style = 'normal'

show(column(select_p3, p3))

We have used a strip plot with a dropdown box that changes between daily caloric intake and proportion of calories that are protein. We used Custom JS, where we first read the option picked from the user and show the corresponding graph based on if the option chosen is calories or protein. 

As we can see, participants with no diabetes have a higher daily energy calorie intake. Similar to the sugar intake plot, this could potentially be in response to diabetes diagnosis for diabetes participants. Upon diagnosis, participants may have been advised to reduce calories, which would explain the decreased intake we see here. Interestingly, borderline/prediabetic participants have the lowest daily calorie intake, which similarily to diabetic participants, may be in result to treatment protocols if they have been identified as prediabetic, and thus may have been advised to reduce calories to prevent type 2 diabetes. 

We can also see for protein/calorie ratio that the no diabetes group has a higher protein intake compared to borderline and diabetic participants. While protein leads to satiety, it seems that diabetic participants have either not been recommended an increase in protein intake or they are not adhering to nutritional advice. We can see that while majority of the non diabetic participants have a protein/calorie ratio of 0.1 - 0.2, similar to the diabetic participants, there is a large group of non diabetic participants that have protein/calorie ratios above 2.5. 


Now, we are going to investigate activity levels and diabetes. In particular, we are going to investigate sedentary minutes and diabetes. 

In [ ]:
from bokeh.models import Select

# Select the columns we need and drop missing glycemic status
p4_df = df[['PAD680', 'PAD790Q', 'PAD800', 'PAD810Q', 'PAD820',
            'glycemic_status']].dropna(subset=['glycemic_status'])

# Calculate weekly activity minutes
p4_df = p4_df.copy()
p4_df['mod_weekly'] = p4_df['PAD790Q'] * p4_df['PAD800']
p4_df['vig_weekly'] = p4_df['PAD810Q'] * p4_df['PAD820']

# Aggregate means by glycaemic status for each measure
status_order = ['Normal', 'Prediabetic', 'Diabetic']
colors       = ['#2ecc71', '#f39c12', '#e74c3c']

def get_means(col):
    grp = p4_df.dropna(subset=[col]).groupby('glycemic_status')[col].mean()
    return [float(grp.get(s, 0)) for s in status_order]

# Pre-compute the three sets of bar heights
means_sed = get_means('PAD680')
means_mod = get_means('mod_weekly')
means_vig = get_means('vig_weekly')

# Build the ColumnDataSource — we store all three in the source
# and swap which column is shown using CustomJS
src_p4 = ColumnDataSource(dict(
    status  = status_order,
    y       = means_sed,        # start showing sedentary
    sed     = means_sed,
    mod     = means_mod,
    vig     = means_vig,
    color   = colors,
))

# Figure
p4 = figure(
    x_range=status_order,
    width=650, height=400,
    title='Mean Daily Sedentary Minutes by Glycaemic Status',
    x_axis_label='Glycaemic status',
    y_axis_label='Mean sedentary minutes per day',
    tools='pan,wheel_zoom,reset,save',
)

p4.vbar(
    x='status', top='y', source=src_p4,
    width=0.5, color='color',
    line_color='white', alpha=0.85,
)

p4.add_tools(HoverTool(tooltips=[
    ('Glycaemic status', '@status'),
    ('Mean value',       '@y{0.0f}'),
]))

# Dropdown to switch between the three measures
select_p4 = Select(
    title='Select measure:',
    value='Sedentary minutes/day',
    options=['Sedentary minutes/day', 'Moderate activity (min/week)', 'Vigorous activity (min/week)'],
    width=280,
)

# CustomJS: when dropdown changes, swap the y values and update axis label + title
cb_p4 = CustomJS(
    args=dict(src=src_p4, select=select_p4, yaxis=p4.yaxis[0], title=p4.title),
    code="""
        const choice = select.value
        if (choice === 'Sedentary minutes/day') {
            src.data['y'] = src.data['sed']
            yaxis.axis_label = 'Mean sedentary minutes per day'
            title.text = 'Mean Daily Sedentary Minutes by Glycaemic Status'
        } else if (choice === 'Moderate activity (min/week)') {
            src.data['y'] = src.data['mod']
            yaxis.axis_label = 'Mean moderate activity minutes per week'
            title.text = 'Mean Weekly Moderate Activity Minutes by Glycaemic Status'
        } else {
            src.data['y'] = src.data['vig']
            yaxis.axis_label = 'Mean vigorous activity minutes per week'
            title.text = 'Mean Weekly Vigorous Activity Minutes by Glycaemic Status'
        }
        src.change.emit()
    """
)
select_p4.js_on_change('value', cb_p4)

show(column(select_p4, p4))

Now, we can see the prevelance of diabetes based on the mean daily sedentary minutes, mean moderate activity and mean vigorous activity based on glycaemic status. As we can see, there is not much of a difference between non diabetic and diabetic participants. We can see that prediabetic participants have the lowest mean daily sedentary minutes with 351, while non diabetic participants have mean daily sedentary minutes of 374, and diabetic have a mean sedentary daily minutes 381. There is not much of a difference between these, even though we would expect that diabetic individuals may have decreased sedentary times based on recommendations upon diagnosis. 

For moderate activity, we can see that non diabetic participants have the highest weekly moderate activity minutes with 224, while prediabetic is at 224, and diabetic is at 201 minutes per week. This result may indicate the relationship between moderate activity and prevention of diabetes, however as this is a cross-sectional study, we can not fully investigate the relationhsip between activity and diabetes. 

Interestingly, we see an inverse with vigourous activity per week, as diabetic individuals have the highest mean value with 162 minutes of weekly vigourous activity, while prediabetic participants also have 162 minutes and diabetic participants have the lowest average with 160. There is a negligible difference between these, and thus it is hard to draw conclusions based on this result.

Based on all 3 of these graphs, we can see that moderate activity levels observed the biggets difference between our participant groups. This result is surprising, as we would expect sedentary minutes or vigourous activity to be the biggest variation between groups. However, it seems that either activity levels have little to no impact on diabetes, or participants that have been diagnosed with diabetes have decreased sedentary levels and increased vigourous activity. This indicates that diabetes treatment may be focused on decreasing sedentary levels and increasing vigorous activity levels with little focus on moderate activity. As stated previously, as this is a cross sectional study, we can only make assumptions on how diabetes has impacted physical activity levels, and not on how physical activity impacts diabetes. 

Finally, we investigate the relationship between income and glycaemic outcomes. We will be using he NHANES poverty-to-income ratio (INDFMPIR), which is a continuous measure comparing household income to the federal poverty line. We will plot this ratio against HbA1c for each participant, coloured by glycaemic status. 

In [ ]:
# Select the columns we need and drop missing values
p5_df = df[['INDFMPIR', 'LBXGH', 'glycemic_status', 'occupation', 'RIDAGEYR', 'gender']].dropna()

# Store full raw data
src_p5_raw = ColumnDataSource(dict(
    income = p5_df['INDFMPIR'].tolist(),
    hba1c  = p5_df['LBXGH'].tolist(),
    status = p5_df['glycemic_status'].tolist(),
    occ    = p5_df['occupation'].tolist(),
    age    = p5_df['RIDAGEYR'].tolist(),
    gender = p5_df['gender'].tolist(),
))

# The displayed source starts as a copy of the full data
src_p5 = ColumnDataSource(dict(
    income = p5_df['INDFMPIR'].tolist(),
    hba1c  = p5_df['LBXGH'].tolist(),
    status = p5_df['glycemic_status'].tolist(),
    occ    = p5_df['occupation'].tolist(),
    age    = p5_df['RIDAGEYR'].tolist(),
    gender = p5_df['gender'].tolist(),
    color  = p5_df['glycemic_status'].map(
                 {'Normal': '#2ecc71', 'Prediabetic': '#f39c12', 'Diabetic': '#e74c3c'}
             ).tolist(),
))

p5 = figure(
    width=680, height=420,
    title='HbA1c vs Income (Poverty Ratio) — All Occupations, All Income Levels',
    x_axis_label='Income-to-poverty ratio (higher = wealthier)',
    y_axis_label='HbA1c (%)',
    tools='pan,wheel_zoom,reset,save',
)

p5.scatter(
    x='income', y='hba1c', source=src_p5,
    color='color', alpha=0.5, size=6,
    legend_field='status',
)

p5.add_tools(HoverTool(tooltips=[
    ('Glycaemic status',   '@status'),
    ('HbA1c (%)',          '@hba1c{0.1f}'),
    ('Poverty ratio',      '@income{0.2f}'),
    ('Occupation',         '@occ'),
    ('Age',                '@age'),
    ('Gender',             '@gender'),
]))

p5.legend.title        = 'Glycaemic status'
p5.legend.location     = 'top_right'
p5.legend.click_policy = 'mute'
p5.background_fill_color = '#f9f9f9'
p5.grid.grid_line_color  = 'white'
p5.title.text_font_size  = '13px'

# --- Controls ---

# Dropdown: filter by occupation (or show all)
occ_options = ['All occupations'] + sorted(p5_df['occupation'].dropna().unique().tolist())
select_occ = Select(
    title='Filter by occupation:',
    value='All occupations',
    options=occ_options,
    width=220,
)

# Slider: filter by income range
income_slider = RangeSlider(
    title='Income range (poverty ratio)',
    start=0.0, end=5.0,
    value=(0.0, 5.0),
    step=0.1,
    width=340,
)

# CustomJS: filters raw data by occupation + income range, updates displayed source
cb_p5 = CustomJS(
    args=dict(
        src=src_p5,
        raw=src_p5_raw,
        select=select_occ,
        slider=income_slider,
        title=p5.title,
    ),
    code="""
        const occ_choice = select.value
        const [lo, hi]   = slider.value

        const color_map = {
            'Normal':      '#2ecc71',
            'Prediabetic': '#f39c12',
            'Diabetic':    '#e74c3c',
        }

        const new_income = [], new_hba1c  = [], new_status = []
        const new_occ    = [], new_age    = [], new_gender = []
        const new_color  = []

        for (let i = 0; i < raw.data['income'].length; i++) {
            const inc = raw.data['income'][i]
            const occ = raw.data['occ'][i]

            // Apply occupation filter
            if (occ_choice !== 'All occupations' && occ !== occ_choice) continue

            // Apply income range filter
            if (inc < lo || inc > hi) continue

            new_income.push(inc)
            new_hba1c.push(raw.data['hba1c'][i])
            new_status.push(raw.data['status'][i])
            new_occ.push(occ)
            new_age.push(raw.data['age'][i])
            new_gender.push(raw.data['gender'][i])
            new_color.push(color_map[raw.data['status'][i]] || '#aaaaaa')
        }

        src.data = {
            income: new_income, hba1c: new_hba1c, status: new_status,
            occ: new_occ, age: new_age, gender: new_gender, color: new_color,
        }
        src.change.emit()

        const occ_label = occ_choice === 'All occupations' ? 'All Occupations' : occ_choice
        title.text = 'HbA1c vs Income (Poverty Ratio) — ' + occ_label +
                     ', Income ' + lo.toFixed(1) + '–' + hi.toFixed(1)
    """
)

select_occ.js_on_change('value', cb_p5)
income_slider.js_on_change('value', cb_p5)

show(column(row(select_occ, income_slider), p5))

For our last plot, we have combined all of the tools from previous plots to create a scatter plot with a dropdown and a slider. The dropdown will allow for selecting between different occupation types, while the slider will allow for sliding between income range (poverty ratio from the demographics table). 

As we can see, for employed participants, we can see there is a fairly even spread in terms of glycaemic status and income range. For non diabetic participants, there is a very even spread throughout with a large cluster at the 5 range. For diabetic participants, there was an even spread with no trend observed in terms of HbA1c percentage and income range. 

For those not working, we can see that majority of these individuals are non diabetic paricipants. There are a few diabetic participants, with some observed with extremely high HbA1c percentages, but there are still majority non diabetes participants here. 

For retired participants, we can see a small cluster of diabetic participants from the 0 to 2 income range, with a high amount of participants with HbA1c percentages above 10. As income range increases, we can see that the number of diabetes participants and HbA1c generally seems to decrease, indicating that retired participants with a high income rating have a lower prevelance of diabetes. However, this can only be assumed and the graph does not supply strong enough evidence to make this conclusion. Again, we see an even spread of non diabetic participants across all income ranges. There is also another cluster of all participants at the 5 income range. 

When looking at those seeking work, we can see there is a low number of participants with diabetes. However, we can see that there is a high proportion of diabetes participants here that have a high HbA1c percentage. In the other graphs, we can see that majority of diabetes participants have low HbA1c percentage, but here we have a high percentage who have a high HbA1c. This may indicate that those who are seeking a job are more at risk of having a dangerous HbA1c. Again, we see a relative spread of those without diabetes.

When looking at the full picture, we can see there is a very even spread of diabetes participants across all types of occupations. It would have been expectes that those with a high income range would have a decreased prevelance of diabetes as they would have better access to healthcare and resources, however this is not necessarily the case here. While we do see a slight decrease in HbA1c as we increase income range, there is not a relatively strong relationship. This graph indicates that income and occupation status ay not have a large impact on diabetes prevelance. 

In conclusion, we have performed analysis on the NHANES dataset to observe the impact of certain variables on diabetes prevelance, including nutrition, activity levels and income. For nutrition, we found that there may be a slight decrease in sugar/carbohydrate ratio in diabetes participants compared to those without diabetes. In terms of fiber, we found that there is a higher number of non diabetic participants that consume a high fiber diet, while there were only a small proportion of diabetic participants that were following the recommended guideline of 25-38g of fiber per day. For protein and calories, we found that there was a larger average consumption in calories in non diabetic participants compared to diabetic participants, but there was a higher proportion of protein consumed in non diabetic participants compared to diabetic participants. For activity levels, we surprisingly found no significant difference in levels between all groups except for moderate activity, where non diabetic participants had a higher number of moderate activity minutes per week on average. In terms of income, we did not find a significant pattern in terms of income range and occupation type. We found the most interesting results in the population seeking jobs, where there was a high proportion of diabetic participants with high HbA1c percentages. 

This report contains analysis on data collected from the NHANES datasets from 2021-2023. This information that we analysed was collected from a single point in time, such as nutritional information and activity levels, and thus we were only able to analyse data in terms of how these variables differ between diabetic and non diabetic participants. To be able to strengthen this analysis and obtain stronger observations, we would need data from an extended period of time to assess how these affect certain health outcomes. Being able to analyse how nutrition and activity levels affect health markers over a certain period of time would provide insightful observations on which variables impact diabetes the most, and this would help in providing future recommendations for those diagnosed with diabetes. 